In [21]:
import pandas as pd
import numpy as np

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [22]:
customer_churn = pd.read_csv(
    'customer_churn.csv'
)

customer_cltv = pd.read_csv(
    'customer_cltv.csv'
)

customer_predictions = pd.read_csv(
    'customer_churn_predictions.csv'
)

print("Datasets Loaded Successfully")

Datasets Loaded Successfully


# Customer Health Index

In [23]:
health_df = pd.merge(
    customer_churn,
    customer_predictions[
        [
            'customer_unique_id',
            'Churn_Probability'
        ]
    ],
    on='customer_unique_id'
)

health_df = pd.merge(
    health_df,
    customer_cltv[
        [
            'customer_unique_id',
            'CLTV'
        ]
    ],
    on='customer_unique_id'
)

health_df.head()

,customer_unique_id,order_purchase_timestamp,Recency,Churn,Risk_Level,revenue,Health_Score,Health_Category,Recommended_Action,Churn_Probability,CLTV
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:00,161,1,High Risk,0.0,79.17,Excellent,Retention Offer,1.0,0.0
1,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:00,586,1,Critical Risk,69.0,24.19,Poor,Win-Back Campaign,1.0,69.0
2,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:00,337,1,Critical Risk,0.0,56.40,Good,Win-Back Campaign,1.0,0.0
3,00053a61a98854899e70ed204dd4bafe,2018-02-28 11:15:00,232,1,Critical Risk,382.0,69.99,Good,Win-Back Campaign,1.0,382.0
4,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03-12 15:22:00,220,1,Critical Risk,104.9,71.54,Good,Win-Back Campaign,1.0,104.9


In [24]:
health_df['Customer_Health_Index'] = (

    (health_df['Health_Score'] * 0.4)

    +

    (
        (100 - (health_df['Churn_Probability'] * 100))
        * 0.4
    )

    +

    (
        (
            health_df['CLTV']
            /
            health_df['CLTV'].max()
        ) * 100
        * 0.2
    )

)

health_df[
    'Customer_Health_Index'
] = health_df[
    'Customer_Health_Index'
].round(2)

health_df[
    [
        'customer_unique_id',
        'Customer_Health_Index'
    ]
].head()

,customer_unique_id,Customer_Health_Index
0,0000366f3b9a7992bf8c76cfdf3221e2,31.67
1,0000f46a3911fa3c0805444483337064,9.78
2,0004aac84e0df4da2b147fca70cf8255,22.56
3,00053a61a98854899e70ed204dd4bafe,28.56
4,0005ef4cd20d2893f0d9fbd94d3c0d97,28.77


# Dynamic Risk Score

In [25]:
health_df['Dynamic_Risk_Score'] = (
    100
    -
    health_df[
        'Customer_Health_Index'
    ]
)

health_df[
    'Dynamic_Risk_Score'
] = health_df[
    'Dynamic_Risk_Score'
].round(2)

health_df[
    [
        'customer_unique_id',
        'Dynamic_Risk_Score'
    ]
].head()

,customer_unique_id,Dynamic_Risk_Score
0,0000366f3b9a7992bf8c76cfdf3221e2,68.33
1,0000f46a3911fa3c0805444483337064,90.22
2,0004aac84e0df4da2b147fca70cf8255,77.44
3,00053a61a98854899e70ed204dd4bafe,71.44
4,0005ef4cd20d2893f0d9fbd94d3c0d97,71.23


# Customer Personas

In [26]:
vip_threshold = (
    health_df['CLTV']
    .quantile(0.75)
)

print(vip_threshold)

138.0


In [27]:
conditions = [

    health_df['CLTV'] > vip_threshold,

    health_df['Customer_Health_Index'] > 80,

    health_df['Dynamic_Risk_Score'] > 80,

    health_df['Customer_Health_Index'] < 40

]

choices = [

    'VIP Customer',

    'Loyal Customer',

    'At Risk Customer',

    'Inactive Customer'

]

health_df['Customer_Persona'] = np.select(
    conditions,
    choices,
    default='Regular Customer'
)

health_df['Customer_Persona'].value_counts()

,count
Customer_Persona,
Inactive Customer,32698
VIP Customer,17278
At Risk Customer,13945
Regular Customer,5206


# Best Action Engine

In [28]:
def next_action(persona):

    if persona == 'Elite VIP':
        return 'Exclusive Premium Rewards'

    elif persona == 'VIP Customer':
        return 'VIP Loyalty Benefits'

    elif persona == 'Loyal Customer':
        return 'Retention Rewards'

    elif persona == 'At Risk Customer':
        return 'Urgent Retention Campaign'

    elif persona == 'Inactive Customer':
        return 'Win-Back Campaign'

    else:
        return 'Regular Marketing'


health_df['Next_Best_Action'] = (
    health_df['Customer_Persona']
    .apply(next_action)
)

health_df[
    [
        'Customer_Persona',
        'Next_Best_Action'
    ]
].head()

,Customer_Persona,Next_Best_Action
0,Inactive Customer,Win-Back Campaign
1,At Risk Customer,Urgent Retention Campaign
2,Inactive Customer,Win-Back Campaign
3,VIP Customer,VIP Loyalty Benefits
4,Inactive Customer,Win-Back Campaign


# Strategic Customer Ranking

In [29]:
top_customers = (
    health_df
    .sort_values(
        'Customer_Health_Index',
        ascending=False
    )
)

top_customers[
    [
        'customer_unique_id',
        'Customer_Health_Index',
        'Customer_Persona'
    ]
].head(20)

,customer_unique_id,Customer_Health_Index,Customer_Persona
53906,c7794313e2cd53472b116a751350f132,83.03,VIP Customer
64274,edde2314c6c30e864a128ac95d6b2112,82.67,VIP Customer
632,025cf7c2f32536f0be1ab412bb6602d2,81.91,VIP Customer
54667,ca27f3dac28fb1063faddd424c9d95fa,81.85,VIP Customer
55747,ce3fe361f9e68bf0a813baaae1334d01,81.30,VIP Customer
67120,f886a3f43af9ac928fbb4b56436c528c,81.15,VIP Customer
39081,910f0afb84bc778e856d3683c7e8a46a,80.47,VIP Customer
42557,9db254a87951220e74c58549749cf128,80.37,VIP Customer
54141,c8460e4251689ba205045f3ea17884a1,80.08,VIP Customer
37329,8a8ac60fd8ea25925f379a33ef277987,80.05,VIP Customer


# Customer Influence Score

In [30]:
health_df['Customer_Influence_Score'] = (

    (
        health_df['CLTV']
        /
        health_df['CLTV'].max()
    ) * 50

    +

    (
        health_df['Customer_Health_Index']
        * 0.5
    )

)

health_df[
    'Customer_Influence_Score'
] = (
    health_df[
        'Customer_Influence_Score'
    ].round(2)
)

health_df[
    [
        'customer_unique_id',
        'Customer_Influence_Score'
    ]
].head()

,customer_unique_id,Customer_Influence_Score
0,0000366f3b9a7992bf8c76cfdf3221e2,15.84
1,0000f46a3911fa3c0805444483337064,5.15
2,0004aac84e0df4da2b147fca70cf8255,11.28
3,00053a61a98854899e70ed204dd4bafe,15.70
4,0005ef4cd20d2893f0d9fbd94d3c0d97,14.78


# Strategic Value Score

In [31]:
health_df['Strategic_Value_Score'] = (

    health_df[
        'Customer_Influence_Score'
    ] * 0.6

    +

    health_df[
        'Customer_Health_Index'
    ] * 0.4

)

health_df[
    'Strategic_Value_Score'
] = (
    health_df[
        'Strategic_Value_Score'
    ].round(2)
)

health_df[
    [
        'customer_unique_id',
        'Strategic_Value_Score'
    ]
].head()

,customer_unique_id,Strategic_Value_Score
0,0000366f3b9a7992bf8c76cfdf3221e2,22.17
1,0000f46a3911fa3c0805444483337064,7.00
2,0004aac84e0df4da2b147fca70cf8255,15.79
3,00053a61a98854899e70ed204dd4bafe,20.84
4,0005ef4cd20d2893f0d9fbd94d3c0d97,20.38


# Retention Priority Score

In [32]:
health_df['Retention_Priority_Score'] = (

    health_df[
        'Dynamic_Risk_Score'
    ] * 0.7

    +

    (
        100
        -
        health_df[
            'Customer_Health_Index'
        ]
    ) * 0.3

)

health_df[
    'Retention_Priority_Score'
] = (
    health_df[
        'Retention_Priority_Score'
    ].round(2)
)

health_df[
    [
        'customer_unique_id',
        'Retention_Priority_Score'
    ]
].head()

,customer_unique_id,Retention_Priority_Score
0,0000366f3b9a7992bf8c76cfdf3221e2,68.33
1,0000f46a3911fa3c0805444483337064,90.22
2,0004aac84e0df4da2b147fca70cf8255,77.44
3,00053a61a98854899e70ed204dd4bafe,71.44
4,0005ef4cd20d2893f0d9fbd94d3c0d97,71.23


# Customer Lifecycle Stage

In [33]:
conditions = [

    health_df[
        'Customer_Health_Index'
    ] > 85,

    health_df[
        'Customer_Health_Index'
    ] > 70,

    health_df[
        'Customer_Health_Index'
    ] > 50,

    health_df[
        'Customer_Health_Index'
    ] > 30

]

choices = [

    'VIP Stage',

    'Loyal Stage',

    'Growth Stage',

    'Declining Stage'

]

health_df[
    'Lifecycle_Stage'
] = np.select(
    conditions,
    choices,
    default='Churn Risk Stage'
)

health_df[
    'Lifecycle_Stage'
].value_counts()

,count
Lifecycle_Stage,
Churn Risk Stage,46597
Declining Stage,15708
Loyal Stage,6821
Growth Stage,1


# Customer 360 Profile

In [34]:
customer_360 = health_df[
    [
        'customer_unique_id',

        'CLTV',

        'Customer_Health_Index',

        'Dynamic_Risk_Score',

        'Customer_Persona',

        'Next_Best_Action',

        'Customer_Influence_Score',

        'Strategic_Value_Score',

        'Retention_Priority_Score',

        'Lifecycle_Stage'
    ]
]

customer_360.head()

,customer_unique_id,CLTV,Customer_Health_Index,Dynamic_Risk_Score,Customer_Persona,Next_Best_Action,Customer_Influence_Score,Strategic_Value_Score,Retention_Priority_Score,Lifecycle_Stage
0,0000366f3b9a7992bf8c76cfdf3221e2,0.0,31.67,68.33,Inactive Customer,Win-Back Campaign,15.84,22.17,68.33,Declining Stage
1,0000f46a3911fa3c0805444483337064,69.0,9.78,90.22,At Risk Customer,Urgent Retention Campaign,5.15,7.00,90.22,Churn Risk Stage
2,0004aac84e0df4da2b147fca70cf8255,0.0,22.56,77.44,Inactive Customer,Win-Back Campaign,11.28,15.79,77.44,Churn Risk Stage
3,00053a61a98854899e70ed204dd4bafe,382.0,28.56,71.44,VIP Customer,VIP Loyalty Benefits,15.70,20.84,71.44,Churn Risk Stage
4,0005ef4cd20d2893f0d9fbd94d3c0d97,104.9,28.77,71.23,Inactive Customer,Win-Back Campaign,14.78,20.38,71.23,Churn Risk Stage


# AI Customer Advisor

In [35]:
print(
    "AI CUSTOMER ADVISOR"
)

print(
    "----------------------"
)

print(
    "Focus on high retention priority customers."
)

print(
    "Protect Elite VIP customers through loyalty programmes."
)

print(
    "Launch win-back campaigns for inactive customers."
)

print(
    "Increase engagement for at-risk customers."
)

print(
    "Prioritise customers with high strategic value."
)

AI CUSTOMER ADVISOR
----------------------
Focus on high retention priority customers.
Protect Elite VIP customers through loyalty programmes.
Launch win-back campaigns for inactive customers.
Increase engagement for at-risk customers.
Prioritise customers with high strategic value.


# Executive Health Dashboard Dataset

In [36]:
dashboard_summary = pd.DataFrame({

    'Metric': [

        'Total Customers',

        'Average Health Index',

        'Average Risk Score',

        'Average Strategic Value',

        'Average Retention Priority'

    ],

    'Value': [

        len(health_df),

        round(
            health_df[
                'Customer_Health_Index'
            ].mean(),
            2
        ),

        round(
            health_df[
                'Dynamic_Risk_Score'
            ].mean(),
            2
        ),

        round(
            health_df[
                'Strategic_Value_Score'
            ].mean(),
            2
        ),

        round(
            health_df[
                'Retention_Priority_Score'
            ].mean(),
            2
        )

    ]

})

dashboard_summary

,Metric,Value
0,Total Customers,69127.00
1,Average Health Index,29.17
2,Average Risk Score,70.83
3,Average Strategic Value,20.68
4,Average Retention Priority,70.83


In [37]:
customer_360.to_csv(
    'customer_360.csv',
    index=False
)

dashboard_summary.to_csv(
    'health_dashboard_summary.csv',
    index=False
)

print(
    "Customer Health Intelligence Datasets Exported Successfully"
)

Customer Health Intelligence Datasets Exported Successfully
